# Phase 1.3: D5 (Qwen2-VL-2B yes/no), smoke test, then stratified subsample (Colab, **T4**)
Part A smoke-tests D5 on the 500-image dev subset and applies Isaac's pre-authorised gate
(2026-11-02): mean yes/no token mass ≥ 0.9 on both datasets, minors flagged more often than adults
on both, and no run errors. Only if ALL pass does part B draw the pre-specified stratified subsample
(≤2,000 per dataset, seed 0) and run D5 on it. Resumable. The gate result goes to Drive.

**Ethics:** never display images; per-image outputs stay on Drive.

In [ ]:
PREAUTHORISED = True    # Isaac 2026-11-02: proceed to part B iff the part A gate passes
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, shutil
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/child-detection-audit')
REPO = pathlib.Path('/content/child-detection-skin-tone-audit')
os.environ['CHILD_AUDIT_ENV'] = 'colab'
os.chdir('/content')
!rm -rf {REPO} && git clone -q https://github.com/isaacmutuma/child-skintone-detection.git {REPO}
!git -C {REPO} log --oneline -1
os.chdir(REPO)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU: switch runtime to T4'
!pip -q install -U "transformers>=4.45" accelerate huggingface_hub
import transformers; print('transformers', transformers.__version__)

## A. D5 smoke test on the dev subset

In [ ]:
!rm -rf /content/dev && mkdir -p /content/dev && unzip -q {DRIVE_ROOT}/dev_subset.zip -d /content/dev
env = dict(os.environ, UTKFACE_DIR='/content/dev/UTKFace', FAIRFACE_DIR='/content/dev/fairface')
import subprocess
for ds in ['utkface', 'fairface_val']:
    print(f'=== vlm / {ds} (dev)')
    subprocess.run(['python', 'scripts/run_detector.py', '--detector', 'vlm', '--dataset', ds,
                    '--dev', '--batch-size', '8'], env=env, check=True)
subprocess.run(['python', 'scripts/smoke_summary.py', '--tag', 'dev'], env=env, check=True)
import pandas as pd
gate = {'preauthorised': PREAUTHORISED, 'checks': {}}
for ds in ['utkface', 'fairface_val']:
    df = pd.read_csv(DRIVE_ROOT / 'per_image' / f'per_image__vlm__{ds}__dev.csv')
    fm, fa = df[df.role=='minor'].is_child.mean(), df[df.role=='adult_fpr'].is_child.mean()
    c = {'n': int(len(df)), 'mean_yesno_mass': float(df.yesno_mass.mean()),
         'min_yesno_mass': float(df.yesno_mass.min()), 'flag_rate_minor': float(fm),
         'flag_rate_adult': float(fa), 'n_fail': int(df.fail_reason.notna().sum())}
    c['pass'] = bool(c['mean_yesno_mass'] >= 0.9 and fm > fa and c['n_fail'] == 0 and c['n'] == 250)
    gate['checks'][ds] = c
    print(ds, c)
gate['all_pass'] = all(c['pass'] for c in gate['checks'].values())
RUN_SUBSAMPLE = PREAUTHORISED and gate['all_pass']
print('D5 GATE:', 'PASS -> running subsample' if RUN_SUBSAMPLE else 'FAIL -> stopping before subsample')
(DRIVE_ROOT / 'results' / 'phase1').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'results' / 'phase1' / 'd5_gate_dev.json').write_text(json.dumps(gate, indent=2))
!mkdir -p {DRIVE_ROOT}/results/phase1/runs && cp results/phase1/runs/vlm__*__dev.json {DRIVE_ROOT}/results/phase1/runs/ && cp results/phase1/smoke_summary_dev.json {DRIVE_ROOT}/results/phase1/smoke_summary_dev_with_vlm.json

## B. Stratified subsample + D5 (only if the part A gate passed)

In [ ]:
if not RUN_SUBSAMPLE:
    print('Gate did not pass (or not pre-authorised) -> D5 subsample NOT run. See d5_gate_dev.json.')
else:
    os.environ.pop('UTKFACE_DIR', None); os.environ.pop('FAIRFACE_DIR', None)
    !python scripts/unpack_data.py
    PER = DRIVE_ROOT / 'per_image'
    if not all((PER / f'vlm_subsample__{ds}.txt').exists() for ds in ['utkface', 'fairface_val']):
        !python scripts/make_vlm_subsample.py
        !cp results/phase1/vlm_subsample.json {DRIVE_ROOT}/results/phase1/
    RUNS_DRIVE = DRIVE_ROOT / 'results' / 'phase1' / 'runs'; RUNS_DRIVE.mkdir(parents=True, exist_ok=True)
    for ds in ['utkface', 'fairface_val']:
        name = f'vlm__{ds}__full_sub.json'
        if (RUNS_DRIVE / name).exists() and (PER / f'per_image__vlm__{ds}__full_sub.csv').exists():
            print(f'skip vlm/{ds} (already on Drive)'); continue
        !python scripts/run_detector.py --detector vlm --dataset {ds} --batch-size 8 --ids-file {PER}/vlm_subsample__{ds}.txt
        d = json.loads((REPO / 'results/phase1/runs' / name).read_text())
        assert d['git']['dirty'] is False
        shutil.copy(REPO / 'results/phase1/runs' / name, RUNS_DRIVE / name)
        print(f"saved {name}: n={d['n_images']} flagged={d['n_flagged_child']} {d['seconds']}s")